# Student Data ETL Pipeline - Analysis Walkthrough

This notebook walks through each stage of the ETL pipeline using modules from `src/`: **Input -> Profile -> Clean & Validate -> Transform -> Load -> Analyze -> Report**. It shows before/after data and generated charts.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

# Ensure project root is on path when running from notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    RAW_DATA_FILE, CLEAN_DATA_FILE, REJECTED_DATA_FILE, SQLITE_DB_FILE, SQLITE_TABLE,
    OUTPUTS_CHARTS_DIR, QUALITY_REPORT_FILE, ANALYSIS_REPORT_FILE
)
from src.extract import extract_data
from src.profile import profile_data
from src.transform import transform_data
from src.load import load_data
from src.analyze import analyze_data
from src.report import generate_quality_report, generate_analysis_report

pd.set_option('display.max_columns', 50)
print('Project root:', PROJECT_ROOT)

## 1. Input (Extract)
Read the raw CSV/Excel file and verify required columns.

In [ ]:
raw_df = extract_data(PROJECT_ROOT / 'data/raw/students_raw.csv' if (PROJECT_ROOT / 'data/raw/students_raw.csv').exists() else RAW_DATA_FILE)
print(f'Raw shape: {raw_df.shape}')
raw_df.head(10)

## 2. Profile (Before Cleaning)
Compute row/null/duplicate counts, out-of-range values, and unique categorical values before any changes.

In [ ]:
raw_profile = profile_data(raw_df)
print('Rows:', raw_profile['row_count'])
print('Total nulls:', raw_profile['total_nulls'])
print('Duplicate Student IDs:', raw_profile['duplicate_student_ids'])
print('Exact duplicate rows:', raw_profile['exact_duplicate_rows'])
print('Out-of-range counts:', raw_profile['out_of_range_counts'])
print('\nNull counts per column:')
display(pd.Series(raw_profile['null_counts'], name='null_count').to_frame())
print('Unique categorical values (sample):')
for k, v in raw_profile['unique_categorical_values'].items():
    print(f'  {k}: {v[:10]}{" ..." if len(v) > 10 else ""}')

## 3. Clean, Validate & Transform
Standardize text/types, resolve duplicates, flag invalid ranges (set to NaN), impute missing attendance with department median, leave missing marks as NaN, and derive fields. Rejected records are kept separately.

In [ ]:
clean_df, rejected_df = transform_data(raw_df)
print(f'Cleaned shape: {clean_df.shape}')
print(f'Rejected shape: {rejected_df.shape}')
print('\nData quality flags:')
display(clean_df['data_quality_flag'].value_counts().to_frame('count'))

### Before vs After (sample rows)
Comparing raw and cleaned views for key columns.

In [ ]:
cols = ['Student ID','Name','Age','Gender','Department','Attendance','Marks']
print('RAW (sample):')
display(raw_df[cols].head(10))
print('CLEANED (sample):')
display(clean_df[[c for c in cols if c in clean_df.columns] + ['Grade_Computed','Performance_Category','Attendance_Band','data_quality_flag']].head(10))
print('REJECTED (sample):')
display(rejected_df.head(10))

## 4. Load
Save cleaned data to CSV and SQLite, save rejected records to CSV, and verify by reading back from SQLite.

In [ ]:
rows, cols_count = load_data(clean_df, rejected_df, CLEAN_DATA_FILE, REJECTED_DATA_FILE, SQLITE_DB_FILE, SQLITE_TABLE)
print(f'Verified load -> rows: {rows}, columns: {cols_count}')
print('Clean CSV:', CLEAN_DATA_FILE)
print('Rejected CSV:', REJECTED_DATA_FILE)
print('SQLite DB:', SQLITE_DB_FILE, 'table:', SQLITE_TABLE)

## 5. Analyze
Compute statistics (averages, correlation, grade distribution, department comparison, top/bottom performers, low attendance) and generate charts.

In [ ]:
results = analyze_data(clean_df, raw_profile, OUTPUTS_CHARTS_DIR)
print('Attendance-Marks correlation:', round(results['attendance_marks_correlation'], 3))
print('Low attendance count (< {}%): {}'.format(results['low_attendance_threshold'], results['low_attendance_count']))
print('\nAverage marks by department:')
display(results['avg_marks_by_department'].to_frame('Average Marks'))
print('Department comparison:')
display(results['department_comparison'])
print('Top performers:')
display(results['top_performers'])
print('Bottom performers:')
display(results['bottom_performers'])

### Charts

In [ ]:
chart_files = [
    'data_quality_issues.png',
    'grade_distribution.png',
    'avg_marks_by_department.png',
    'attendance_vs_marks.png',
    'marks_distribution.png',
]
for name in chart_files:
    path = OUTPUTS_CHARTS_DIR / name
    print(name)
    display(Image(filename=str(path)))

## 6. Report
Generate Markdown reports for data quality (before/after, fixes, rejections) and analysis summary (key findings with numbers from actual data).

In [ ]:
generate_quality_report(raw_profile, clean_df, rejected_df, QUALITY_REPORT_FILE)
generate_analysis_report(results, ANALYSIS_REPORT_FILE)
print('Quality report:', QUALITY_REPORT_FILE)
print('Analysis report:', ANALYSIS_REPORT_FILE)

## Summary
The pipeline produced cleaned CSV/SQLite outputs, logged rejected records, saved charts, and generated reports. All numbers in reports/charts come from the actual processed data.